# Latent-space DDNM (inpainting UNet, CelebA-HQ)

Model: **`stable-diffusion-v1-5/stable-diffusion-inpainting`** — UNet takes **9 channels**: `[noisy latents (4) | mask (1) | masked-image latents (4)]` (same layout as `diffusers` inpaint pipeline).

**Each reverse step** (iterate):

1. **Encode (conditioning + state)** — VAE-encode **once** before the loop: latent **mask** (SD convention: 1 = hole to fill) + **masked image** latents (known pixels = `y`). Every step: **`scheduler.scale_model_input(z_t)`** → **concat** → UNet input.
2. **DDIM / DDNM latent update** — UNet predicts $\varepsilon$; build $z_{0|t}$; after pixel projection, $z_0^{\hat{}}$ and consistent $\hat\varepsilon$; sample $z_{t-1}$.
3. **Decode** — $z_{0|t}$ → $x_{0|t}$ in $[-1,1]$ for the measurement step.
4. **Back-projection** — simplified DDNM null-space inpainting in **pixels**.
5. **Encode** — $x_0^{\hat{}}$ → $z_0^{\hat{}}$ (then folded into step 2’s update).

**Also:** `next_t = -1` must use $\bar\alpha=1$, not `alphas_cumprod[-1]`. After projection, use **$\hat\varepsilon$** so $z_t = \sqrt{\bar\alpha_t} z_0^{\hat{}} + \sqrt{1-\bar\alpha_t}\,\hat\varepsilon$.

**Grids:** (1) **Per-step** — two rows (SD VAE / SPNN): original | masked | subsampled projected $x_0^{hat}$ after each reverse step. (2) **Summary** — original | masked | final SD | final SPNN.

**SPNN checkpoint:** use **`load_spnn256_vae_adapter`** with the same **`SPNN_KWARGS`** as training (`SPNN256_DEFAULT_KWARGS` matches `mix_type="cayley", hidden=192, r_hidden=384, scale_bound=2.0`). The adapter resizes 512↔256 and pads latents 3→4 ch. For native 512 SPNN weights, use **`load_spnn512_vae_adapter`** (optional **`scaling_factor=float(vae.config.scaling_factor)`** so latents match the SD / inpainting UNet; default is SD1.5’s factor). Inpainting does **not** change the 4-channel latent shape — only the UNet adds mask + masked-image latents.

In [ ]:
%pip install -q diffusers transformers accelerate safetensors datasets

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.utils import make_grid

from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer

# Repo root: cwd is DDNM/ when the notebook lives there; otherwise use ./DDNM
DDNM_ROOT = Path.cwd()
if not (DDNM_ROOT / "functions" / "latent_ddnm.py").is_file():
    DDNM_ROOT = Path.cwd() / "DDNM"
if str(DDNM_ROOT) not in sys.path:
    sys.path.insert(0, str(DDNM_ROOT))

from functions.latent_ddnm import latent_ddnm_inpaint_simplified

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32  # use float16 on CUDA for speed if you prefer (load models with torch_dtype=torch.float16)
print("device:", device)

device: cuda


In [5]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"

tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(device, dtype=dtype)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device, dtype=dtype)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device, dtype=dtype)
scheduler = DDIMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

vae.eval()
unet.eval()
text_encoder.eval()

for m in (vae, unet, text_encoder):
    m.requires_grad_(False)

alphas_cumprod = scheduler.alphas_cumprod.to(device)
num_train_timesteps = scheduler.config.num_train_timesteps

# Unconditional (empty prompt) — same embedding used for every step
tokens = tokenizer("", padding="max_length", max_length=tokenizer.model_max_length, return_tensors="pt")
prompt_embeds = text_encoder(tokens.input_ids.to(device))[0]
print("num_train_timesteps:", num_train_timesteps)

# SPNN256 checkpoint (same pattern as training: SPNNAutoencoder256 + kwargs + state_dict)
from spnn_model import SPNNAutoencoder512

SPNN_CKPT = DDNM_ROOT / "spnn_vae_best_512.pt"
print(f"Loading SPNN from {SPNN_CKPT}...")
spnn_vae = SPNNAutoencoder512().to('cpu').eval()
ckpt = torch.load(SPNN_CKPT, map_location='cpu', weights_only=True)
spnn_vae.load_state_dict(ckpt["model_state_dict"])
# Adapter pads 3→4 latent channels + resizes 512↔256 so the SD UNet pipeline still runs
print("SPNN512 VAE loaded from", SPNN_CKPT)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-inpainting
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


num_train_timesteps: 1000
Loading SPNN from /home/ron.libman/spnn/spnn_vae_best_512.pt...
SPNN512 VAE loaded from /home/ron.libman/spnn/spnn_vae_best_512.pt


In [6]:
def load_celeba_hq_val_image(image_size=512, seed=0):
    """Random CelebA-HQ face from Hugging Face Hub `iamivan11/CelebA-HQ-zip` (no Google Drive)."""
    import io
    from datasets import load_dataset
    from PIL import Image

    ds = load_dataset("iamivan11/CelebA-HQ-zip", split="validation")
    g = torch.Generator().manual_seed(seed)
    idx = torch.randint(0, len(ds), (1,), generator=g).item()
    row = ds[idx]
    img = row.get("image")
    if isinstance(img, Image.Image):
        pil = img.convert("RGB")
    elif isinstance(img, dict) and "bytes" in img:
        pil = Image.open(io.BytesIO(img["bytes"])).convert("RGB")
    else:
        raise KeyError(f"Unexpected image field; row keys: {list(row.keys())}")

    to_t = T.Compose([T.Resize((image_size, image_size)), T.ToTensor()])
    x01 = to_t(pil).unsqueeze(0).to(device)
    return x01, idx


def make_center_box_mask(b, c, h, w, box=96, device=torch.device("cpu")):
    m = torch.ones(b, 1, h, w, device=device)
    y0, x0 = (h - box) // 2, (w - box) // 2
    m[:, :, y0 : y0 + box, x0 : x0 + box] = 0
    return m


def to_m11(x01):
    return x01 * 2.0 - 1.0


def to_01(x_m11):
    return (x_m11.clamp(-1, 1) + 1.0) / 2.0


def latent_to_image_01(z_final, vae_module):
    """Decode final latents (diffusers VAE, SPNN adapter, or raw SPNN — no `.config`)."""
    cfg = getattr(vae_module, "config", None)
    sc = float(getattr(cfg, "scaling_factor", 1.0)) if cfg is not None else 1.0
    with torch.no_grad():
        dec = vae_module.decode(z_final.to(device) / sc)
        x_m11 = dec.sample if hasattr(dec, "sample") else dec
    return to_01(x_m11)

In [ ]:
# --- Diagnostic: SD VAE vs SPNN (encode → decode round-trip + latent stats) ---
# Gray outputs often come from wrong scaling vs the UNet, or SPNN latents far from SD's distribution.

def _sd_encode_scaled(x_m11_bchw):
    """Latents in the same space as diffusers / latent_ddnm (scaled)."""
    with torch.no_grad():
        post = vae.encode(x_m11_bchw.to(device))
        z = post.latent_dist.mode()
        return z * float(vae.config.scaling_factor)


def _sd_decode_from_scaled(z_scaled):
    with torch.no_grad():
        return vae.decode(z_scaled / float(vae.config.scaling_factor)).sample


def _spnn_encode_raw(spnn, x_m11_bchw):
    """Raw SPNN latent (no extra scaling — matches latent_ddnm when vae has no .config)."""
    with torch.no_grad():
        return spnn.encode(x_m11_bchw.to(next(spnn.parameters()).device))


def _spnn_decode_raw(spnn, z):
    with torch.no_grad():
        return spnn.decode(z)


torch.manual_seed(0)
x01, _idx = load_celeba_hq_val_image(image_size=512, seed=0)
x_m11 = to_m11(x01)

# Move SPNN to same device as tensors used in the check
spnn_dev = device if torch.cuda.is_available() else next(spnn_vae.parameters()).device
spnn_vae.to(spnn_dev).eval()

z_sd = _sd_encode_scaled(x_m11)
z_spnn = _spnn_encode_raw(spnn_vae, x_m11.to(spnn_dev))

print("--- Latent tensors ---")
print(f"z_sd     shape={tuple(z_sd.shape)}  dtype={z_sd.dtype}  device={z_sd.device}")
print(f"z_spnn   shape={tuple(z_spnn.shape)}  dtype={z_spnn.dtype}  device={z_spnn.device}")
for name, z in [("z_sd (SD scaled space)", z_sd), ("z_spnn (SPNN raw)", z_spnn)]:
    zf = z.float()
    print(f"{name}:  mean={zf.mean().item():.6f}  std={zf.std().item():.6f}  min={zf.min().item():.4f}  max={zf.max().item():.4f}")

# Cosine similarity of flattened latents (informative only if shapes match)
if z_sd.shape == z_spnn.shape:
    a = z_sd.flatten()
    b = z_spnn.flatten().to(z_sd.device)
    cos = (a * b).sum() / (a.norm() * b.norm() + 1e-12)
    print(f"cosine(z_sd, z_spnn): {cos.item():.6f}")
    print(
        f"std ratio z_spnn/z_sd: {(z_spnn.float().std() / (z_sd.float().std() + 1e-12)).item():.4f}"
        "  (often ~1 if same scaling space)"
    )
else:
    print("(shapes differ — cosine skipped)")

# Round-trip images [0,1]
rec_sd = to_01(_sd_decode_from_scaled(z_sd))
rec_spnn = to_01(_spnn_decode_raw(spnn_vae, z_spnn).to(device))

# Cross-decode: if SPNN latents live in the same space as SD, SD decode(z_spnn / sc) ≈ image
sc_vae = float(vae.config.scaling_factor)
cross_sd_from_spnn = None
if z_spnn.shape == z_sd.shape:
    with torch.no_grad():
        cross_sd_from_spnn = to_01(vae.decode(z_spnn.to(device) / sc_vae).sample)

fig, ax = plt.subplots(2, 3, figsize=(11, 7))
ax[0, 0].imshow(x01[0].cpu().permute(1, 2, 0).numpy())
ax[0, 0].set_title("input")
ax[0, 1].imshow(rec_sd[0].cpu().permute(1, 2, 0).numpy())
ax[0, 1].set_title("SD VAE recon (z_sd)")
ax[0, 2].imshow(rec_spnn[0].cpu().permute(1, 2, 0).numpy())
ax[0, 2].set_title("SPNN recon (z_spnn)")
err = (rec_sd - x01.to(rec_sd.device)).abs().mean(dim=1, keepdim=True).repeat(1, 3, 1, 1)
ax[1, 0].imshow(err[0].cpu().permute(1, 2, 0).numpy())
ax[1, 0].set_title("|SD recon − input|")
if cross_sd_from_spnn is not None:
    ax[1, 1].imshow(cross_sd_from_spnn[0].cpu().permute(1, 2, 0).numpy())
    ax[1, 1].set_title("SD VAE decode(z_spnn / sc_vae)\n(same space as SD latents?)")
else:
    ax[1, 1].text(0.5, 0.5, "shape mismatch\n(no cross-decode)", ha="center", va="center")
    ax[1, 1].axis("off")
ax[1, 2].imshow((rec_spnn - rec_sd.to(rec_spnn.device)).abs()[0].cpu().permute(1, 2, 0).numpy())
ax[1, 2].set_title("|SPNN recon − SD recon|")
for a in ax.ravel():
    a.axis("off")
plt.suptitle(
    "Top: round-trips. Bottom mid: if gray, z_spnn is not in SD latent space (try SPNN512VAEAdapter + scaling_factor).",
    y=1.02,
)
plt.tight_layout()
plt.show()


In [7]:
# --- Same hyperparameters as DDNM configs/imagenet_256.yml (time_travel) ---
T_sampling = 10
travel_length = 1
travel_repeat = 1
eta = 0.85
sigma_y = 0.0

# 512 matches SD1.5 training (latent 64×64); 256 works but is off-distribution for the UNet.
image_size = 512
torch.manual_seed(42)

x01, img_idx = load_celeba_hq_val_image(image_size=image_size, seed=42)
x_m11 = to_m11(x01)

mask = make_center_box_mask(1, 3, image_size, image_size, box=192, device=device)
mask3 = mask.expand(-1, 3, -1, -1)

# Degradation y = A(x) = x * mask (known pixels); same as simplified DDNM inpainting
y_m11 = mask3 * x_m11

# Latent noise (4×64×64 — SD VAE; SPNN256 adapter pads 3→4 ch at same spatial size)
b, c, h, w = 1, 4, image_size // 8, image_size // 8
z0 = torch.randn(b, c, h, w, device=device, dtype=dtype)

# Same initial noise for both runs; only the encode/decode (first stage) differs
z_out_sd, snaps_sd = latent_ddnm_inpaint_simplified(
    z0.clone(),
    y_m11,
    mask,
    unet,
    vae,
    scheduler,
    prompt_embeds,
    alphas_cumprod,
    num_train_timesteps=num_train_timesteps,
    T_sampling=T_sampling,
    travel_length=travel_length,
    travel_repeat=travel_repeat,
    eta=eta,
    sigma_y=sigma_y,
    device=device,
    return_snapshots=True,
)

z_out_spnn, snaps_spnn = latent_ddnm_inpaint_simplified(
    z0.clone(),
    y_m11,
    mask,
    unet,
    spnn_vae,
    scheduler,
    prompt_embeds,
    alphas_cumprod,
    num_train_timesteps=num_train_timesteps,
    T_sampling=T_sampling,
    travel_length=travel_length,
    travel_repeat=travel_repeat,
    eta=eta,
    sigma_y=sigma_y,
    device=device,
    return_snapshots=True,
)

inpainted_sd_01 = latent_to_image_01(z_out_sd, vae)
inpainted_spnn_01 = latent_to_image_01(z_out_spnn, spnn_vae)
masked_vis_01 = to_01(y_m11)

print("CelebA-HQ validation index:", img_idx)

latent DDNM (inpaint-UNet): 100%|██████████| 10/10 [00:00<00:00, 14.14it/s]


RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [ ]:
# Per-step projected estimate x0_hat (after DDNM null-space step): row 1 = SD VAE, row 2 = SPNN
# Columns: original | masked (y) | subsampled steps (full run has len(snaps_*) steps)
MAX_PROGRESS = 23  # original + masked + up to 20 snapshot columns (adjust if figure is too wide)
n_snap = len(snaps_sd)
k = min(MAX_PROGRESS - 2, n_snap)
idxs = np.linspace(0, n_snap - 1, k, dtype=int) if n_snap > 0 else np.array([], dtype=int)

masked_cpu = masked_vis_01.cpu()
cols_sd = [x01.cpu(), masked_cpu] + [snaps_sd[i] for i in idxs]
cols_sp = [x01.cpu(), masked_cpu] + [snaps_spnn[i] for i in idxs]
n_col = len(cols_sd)

row_sd = make_grid(torch.cat(cols_sd, dim=0), nrow=n_col, padding=4, pad_value=1.0)
row_sp = make_grid(torch.cat(cols_sp, dim=0), nrow=n_col, padding=4, pad_value=1.0)
prog = torch.cat([row_sd, row_sp], dim=1)

plt.figure(figsize=(min(2.2 * n_col, 28), 5))
plt.imshow(prog.permute(1, 2, 0).numpy())
plt.axis("off")
plt.title(
    "Top: SD VAE — Bottom: SPNN  |  col0=original, col1=masked, then subsampled reverse steps (x0_hat each step)"
)
plt.tight_layout()
plt.show()

In [ ]:
grid = make_grid(
    torch.cat([x01, masked_vis_01, inpainted_sd_01, inpainted_spnn_01], dim=0),
    nrow=4,
    padding=8,
    pad_value=1.0,
)
plt.figure(figsize=(16, 4))
plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
plt.axis("off")
plt.title("original  |  masked  |  latent DDNM (SD VAE)  |  latent DDNM (SPNN256, best.pt)")
plt.tight_layout()
plt.show()